# Independent Cross-Series Validation

**Portfolio version.** This notebook validates the primary threshold-based forecast construction against an independently priced Kalshi CPI series expressed as mutually exclusive outcome bins.


# KXECONSTATCPIYOY Cross-Series Consistency Check

Independent validation against Kalshi’s mutually exclusive CPI outcome-bin series. Run top to bottom.

**Requires `01_data_and_forecast_pipeline.ipynb` to have been run first** - Part B below loads `forecast_panel_adjusted.csv`, which Notebook 1 produces.

Note: one function name collided between the two original notebooks (`build_snapshots_for_market`, defined differently in each). It's been renamed to `build_econstat_snapshots_for_market` in Part B below so nothing silently shadows the KXCPIYOY version from Notebook 1 if this file is ever run in the same session.

## PART A - Pull KXECONSTATCPIYOY (the mutually-exclusive bins series)

Reuses the exact same proven live/historical pull logic as the primary KXCPIYOY pipeline, just pointed at the second series.

In [ ]:
import requests
import time
import json
import os
import re
import csv
from datetime import datetime, timezone

BASE_URL = "https://external-api.kalshi.com/trade-api/v2"
SERIES_TICKER = "KXECONSTATCPIYOY"

OUTPUT_DIR = "kalshi_data"
ECONSTAT_DIR = os.path.join(OUTPUT_DIR, "econstat_candlesticks")
REQUEST_DELAY_SECONDS = 0.3

os.makedirs(ECONSTAT_DIR, exist_ok=True)

### Step 1: pull all events (same pagination logic as KXCPIYOY)

In [ ]:
def get_all_events(series_ticker, status="settled"):
    events = []
    cursor = ""
    while True:
        params = {"series_ticker": series_ticker, "status": status, "limit": 200}
        if cursor:
            params["cursor"] = cursor
        resp = requests.get(f"{BASE_URL}/events", params=params)
        resp.raise_for_status()
        data = resp.json()
        events.extend(data.get("events", []))
        cursor = data.get("cursor", "")
        print(f"  pulled {len(data.get('events', []))} events, running total: {len(events)}")
        if not cursor:
            break
        time.sleep(REQUEST_DELAY_SECONDS)
    return events

econstat_events = get_all_events(SERIES_TICKER, status="settled")
print(f"\nTotal KXECONSTATCPIYOY events found: {len(econstat_events)}")

with open(os.path.join(OUTPUT_DIR, "econstat_events.json"), "w") as f:
    json.dump(econstat_events, f, indent=2)

econstat_events[:2]

  pulled 7 events, running total: 7

Total KXECONSTATCPIYOY events found: 7


[{'available_on_brokers': True,
  'category': 'Economics',
  'collateral_return_type': 'MECNET',
  'event_ticker': 'KXECONSTATCPIYOY-26JUN',
  'exchange_index': 0,
  'last_updated_ts': '2026-05-10T00:33:53.100353Z',
  'mutually_exclusive': True,
  'series_ticker': 'KXECONSTATCPIYOY',
  'settlement_sources': [{'name': 'Bureau of Labor Statistics- Consumer Price Index',
    'url': 'https://www.bls.gov/news.release/cpi.nr0.htm'}],
  'strike_period': '',
  'sub_title': 'In Jun 2026',
  'title': 'CPI year-over-year in Jun 2026?'},
 {'available_on_brokers': True,
  'category': 'Economics',
  'collateral_return_type': 'MECNET',
  'event_ticker': 'KXECONSTATCPIYOY-26MAY',
  'exchange_index': 0,
  'last_updated_ts': '2026-05-10T00:33:53.103269Z',
  'mutually_exclusive': True,
  'series_ticker': 'KXECONSTATCPIYOY',
  'settlement_sources': [{'name': 'Bureau of Labor Statistics- Consumer Price Index',
    'url': 'https://www.bls.gov/news.release/cpi.nr0.htm'},
   {'name': 'Bureau of Labor Statisti

### Step 2: markets + date range (live -> historical fallback, exactly as before)

In [ ]:
def get_live_markets(event_ticker):
    params = {"with_nested_markets": "true"}
    resp = requests.get(f"{BASE_URL}/events/{event_ticker}", params=params)
    resp.raise_for_status()
    data = resp.json()
    event = data.get("event", data)
    return event.get("markets", [])


def get_historical_markets(event_ticker):
    params = {"event_ticker": event_ticker, "limit": 200}
    resp = requests.get(f"{BASE_URL}/historical/markets", params=params)
    resp.raise_for_status()
    data = resp.json()
    return data.get("markets", [])


def markets_to_date_range(markets):
    open_times = [m["open_time"] for m in markets if m.get("open_time")]
    close_times = [m["close_time"] for m in markets if m.get("close_time")]
    if not open_times or not close_times:
        return None, None
    def iso_to_unix(s):
        if s.endswith("Z"):
            s = s[:-1] + "+00:00"
        return int(datetime.fromisoformat(s).timestamp())
    start_ts = min(iso_to_unix(t) for t in open_times)
    end_ts = max(iso_to_unix(t) for t in close_times)
    return start_ts - 86400, end_ts + 86400


def get_event_markets_and_range(event_ticker):
    markets = get_live_markets(event_ticker)
    if markets:
        s, e = markets_to_date_range(markets)
        if s is not None:
            return markets, s, e, "live"
    time.sleep(REQUEST_DELAY_SECONDS)
    markets = get_historical_markets(event_ticker)
    if markets:
        s, e = markets_to_date_range(markets)
        if s is not None:
            return markets, s, e, "historical"
    return [], None, None, "none"

### Step 3: candlesticks - live (batched) vs historical (per-market), same as KXCPIYOY

In [ ]:
def get_live_event_candlesticks(series_ticker, event_ticker, start_ts, end_ts, period_interval=1440):
    params = {"start_ts": start_ts, "end_ts": end_ts, "period_interval": period_interval}
    resp = requests.get(
        f"{BASE_URL}/series/{series_ticker}/events/{event_ticker}/candlesticks", params=params,
    )
    resp.raise_for_status()
    return resp.json()


def get_historical_market_candlesticks(ticker, start_ts, end_ts, period_interval=1440):
    params = {"start_ts": start_ts, "end_ts": end_ts, "period_interval": period_interval}
    resp = requests.get(f"{BASE_URL}/historical/markets/{ticker}/candlesticks", params=params)
    resp.raise_for_status()
    return resp.json()


def get_historical_event_candlesticks(markets, start_ts, end_ts, period_interval=1440):
    market_tickers, market_candlesticks = [], []
    for m in markets:
        ticker = m["ticker"]
        data = get_historical_market_candlesticks(ticker, start_ts, end_ts, period_interval)
        market_tickers.append(ticker)
        market_candlesticks.append(data.get("candlesticks", []))
        time.sleep(REQUEST_DELAY_SECONDS)
    return {"market_tickers": market_tickers, "market_candlesticks": market_candlesticks}

In [ ]:
pull_summary_rows = []

for i, event in enumerate(econstat_events, start=1):
    event_ticker = event["event_ticker"]
    print(f"[{i}/{len(econstat_events)}] {event_ticker}")
    try:
        markets, s_ts, e_ts, source = get_event_markets_and_range(event_ticker)
        time.sleep(REQUEST_DELAY_SECONDS)
        if source == "none":
            print("    -> no market data found, skipping")
            continue
        if source == "live":
            candles = get_live_event_candlesticks(SERIES_TICKER, event_ticker, s_ts, e_ts)
        else:
            candles = get_historical_event_candlesticks(markets, s_ts, e_ts)

        out_path = os.path.join(ECONSTAT_DIR, f"{event_ticker}.json")
        with open(out_path, "w") as f:
            json.dump(candles, f, indent=2)

        n_markets = len(candles.get("market_tickers", []))
        pull_summary_rows.append({"event_ticker": event_ticker, "source": source, "n_markets": n_markets})
        print(f"    -> [{source}] {n_markets} markets saved")
    except requests.exceptions.HTTPError as e:
        print(f"    -> HTTP error: {e}")
        pull_summary_rows.append({"event_ticker": event_ticker, "source": "error", "n_markets": 0})

print(f"\nDone. {len(pull_summary_rows)} events processed.")

[1/7] KXECONSTATCPIYOY-26JUN
    -> [live] 26 markets saved
[2/7] KXECONSTATCPIYOY-26MAY
    -> [live] 26 markets saved
[3/7] KXECONSTATCPIYOY-26APR
    -> [historical] 23 markets saved
[4/7] KXECONSTATCPIYOY-26MAR
    -> [historical] 16 markets saved
[5/7] KXECONSTATCPIYOY-26FEB
    -> [historical] 16 markets saved
[6/7] KXECONSTATCPIYOY-26JAN
    -> [historical] 16 markets saved
[7/7] KXECONSTATCPIYOY-25DEC
    -> [historical] 16 markets saved

Done. 7 events processed.


### Step 4: flatten (same close_dollars/close schema fix discovered in Week 3)

In [ ]:
def parse_threshold(market_ticker):
    match = re.search(r"-T(-?\d+\.?\d*)$", market_ticker)
    return float(match.group(1)) if match else None


def get_field(obj, base_name):
    if obj is None:
        return None
    if f"{base_name}_dollars" in obj:
        return obj[f"{base_name}_dollars"]
    if base_name in obj:
        return obj[base_name]
    return None


def flatten_event_file(filepath, event_ticker):
    with open(filepath) as f:
        data = json.load(f)
    rows = []
    for market_ticker, candles in zip(data.get("market_tickers", []), data.get("market_candlesticks", [])):
        threshold = parse_threshold(market_ticker)
        for c in candles:
            ts = c.get("end_period_ts")
            date_str = datetime.fromtimestamp(ts, tz=timezone.utc).strftime("%Y-%m-%d") if ts else None
            yes_bid = get_field(c.get("yes_bid"), "close")
            yes_ask = get_field(c.get("yes_ask"), "close")
            mid_quote = None
            if yes_bid is not None and yes_ask is not None:
                mid_quote = round((float(yes_bid) + float(yes_ask)) / 2, 4)
            rows.append({"event_ticker": event_ticker, "market_ticker": market_ticker,
                         "threshold": threshold, "date": date_str, "mid_quote": mid_quote})
    return rows


all_rows = []
for fname in sorted(os.listdir(ECONSTAT_DIR)):
    if not fname.endswith(".json"):
        continue
    event_ticker = fname.replace(".json", "")
    all_rows.extend(flatten_event_file(os.path.join(ECONSTAT_DIR, fname), event_ticker))

import pandas as pd
econstat_flat = pd.DataFrame(all_rows)
econstat_flat.to_csv(os.path.join(OUTPUT_DIR, "econstat_flattened.csv"), index=False)
print(f"Flattened {len(econstat_flat)} rows -> econstat_flattened.csv")
print("Missing mid_quote:", econstat_flat['mid_quote'].isna().sum(), "of", len(econstat_flat))

Flattened 13551 rows -> econstat_flattened.csv
Missing mid_quote: 0 of 13551


---
# PART B - Full multi-horizon comparison against KXCPIYOY
---

# KXECONSTATCPIYOY Full Cross-Series Consistency Check (Week 4)

**Extends the quick `final`-only version to all 6 horizons.** Reuses the exact proven functions from the pull, flatten, and fixed-horizon-snapshot notebooks - nothing new invented, just applied to the second series and joined against the first.

**Assumes you've already run `kalshi_econstat_cross_check.ipynb` once**, so `econstat_flattened.csv` and `econstat_events.json` already exist in `kalshi_data/`. This notebook adds: econstat market metadata (release dates), econstat fixed-horizon snapshots, and a full multi-horizon comparison against KXCPIYOY.

**If those files don't exist yet**, run the earlier notebook first - this one loads them, it doesn't re-pull raw candlesticks.

In [ ]:
import requests
import time
import json
import os
import re
import csv
import pandas as pd
from datetime import datetime, timezone, timedelta

BASE_URL = "https://external-api.kalshi.com/trade-api/v2"
SERIES_TICKER = "KXECONSTATCPIYOY"
OUTPUT_DIR = "kalshi_data"
REQUEST_DELAY_SECONDS = 0.3
HORIZONS = [21, 14, 7, 3, 1]

econstat_flat = pd.read_csv(os.path.join(OUTPUT_DIR, "econstat_flattened.csv"))
econstat_flat["date"] = pd.to_datetime(econstat_flat["date"]).dt.date

with open(os.path.join(OUTPUT_DIR, "econstat_events.json")) as f:
    econstat_events = json.load(f)

print(f"Loaded {len(econstat_flat)} flattened rows, {len(econstat_events)} events")

Loaded 13551 flattened rows, 7 events


## Build econstat market_metadata (release dates per contract) - reusing the same live/historical fallback

In [ ]:
def get_live_markets(event_ticker):
    params = {"with_nested_markets": "true"}
    resp = requests.get(f"{BASE_URL}/events/{event_ticker}", params=params)
    resp.raise_for_status()
    data = resp.json()
    event = data.get("event", data)
    return event.get("markets", [])


def get_historical_markets(event_ticker):
    params = {"event_ticker": event_ticker, "limit": 200}
    resp = requests.get(f"{BASE_URL}/historical/markets", params=params)
    resp.raise_for_status()
    data = resp.json()
    return data.get("markets", [])


def get_markets_for_event(event_ticker):
    markets = get_live_markets(event_ticker)
    if markets:
        return markets, "live"
    time.sleep(REQUEST_DELAY_SECONDS)
    markets = get_historical_markets(event_ticker)
    return markets, "historical" if markets else "none"


def parse_threshold_from_ticker(market_ticker):
    match = re.search(r"-T(-?\d+\.?\d*)$", market_ticker)
    return float(match.group(1)) if match else None


metadata_rows = []
for i, event in enumerate(econstat_events, start=1):
    event_ticker = event["event_ticker"]
    print(f"[{i}/{len(econstat_events)}] {event_ticker}")
    markets, source = get_markets_for_event(event_ticker)
    time.sleep(REQUEST_DELAY_SECONDS)
    if not markets:
        print("    -> no markets found, skipping")
        continue
    for m in markets:
        threshold = m.get("floor_strike")
        if threshold is None:
            threshold = m.get("cap_strike")
        if threshold is None:
            threshold = parse_threshold_from_ticker(m["ticker"])
        metadata_rows.append({
            "event_ticker": event_ticker, "market_ticker": m["ticker"],
            "threshold": threshold, "close_time": m.get("close_time"),
            "market_source": source,
        })

econstat_meta = pd.DataFrame(metadata_rows)
econstat_meta.to_csv(os.path.join(OUTPUT_DIR, "econstat_market_metadata.csv"), index=False)
print(f"\nBuilt metadata for {len(econstat_meta)} contracts")

[1/7] KXECONSTATCPIYOY-26JUN
[2/7] KXECONSTATCPIYOY-26MAY
[3/7] KXECONSTATCPIYOY-26APR
[4/7] KXECONSTATCPIYOY-26MAR
[5/7] KXECONSTATCPIYOY-26FEB
[6/7] KXECONSTATCPIYOY-26JAN
[7/7] KXECONSTATCPIYOY-25DEC

Built metadata for 139 contracts


## Build econstat fixed-horizon snapshots - reusing the exact function from the KXCPIYOY pipeline

In [ ]:
econstat_meta["release_date"] = pd.to_datetime(econstat_meta["close_time"]).dt.date


def build_econstat_snapshots_for_market(  # renamed from build_snapshots_for_market to avoid colliding with the KXCPIYOY version above
market_ticker, release_date, market_flat_rows):
    rows = market_flat_rows.sort_values("date")
    snapshots = []
    for h in HORIZONS:
        target_date = release_date - timedelta(days=h)
        eligible = rows[rows["date"] <= target_date]
        if eligible.empty:
            snapshots.append({"horizon": f"{h}d", "mid_quote": None, "is_missing": True})
        else:
            r = eligible.iloc[-1]
            snapshots.append({"horizon": f"{h}d", "mid_quote": r["mid_quote"], "is_missing": False})
    final_eligible = rows[rows["date"] < release_date]
    if final_eligible.empty:
        snapshots.append({"horizon": "final", "mid_quote": None, "is_missing": True})
    else:
        r = final_eligible.iloc[-1]
        snapshots.append({"horizon": "final", "mid_quote": r["mid_quote"], "is_missing": False})
    return snapshots


econstat_snap_rows = []
for i, row in econstat_meta.iterrows():
    market_ticker = row["market_ticker"]
    market_flat = econstat_flat[econstat_flat["market_ticker"] == market_ticker]
    if market_flat.empty or pd.isna(row["release_date"]):
        continue
    for s in build_econstat_snapshots_for_market(market_ticker, row["release_date"], market_flat):
        s["event_ticker"] = row["event_ticker"]
        s["market_ticker"] = market_ticker
        s["threshold"] = row["threshold"]
        econstat_snap_rows.append(s)

econstat_snap = pd.DataFrame(econstat_snap_rows)
econstat_snap.to_csv(os.path.join(OUTPUT_DIR, "econstat_fixed_horizon_snapshots.csv"), index=False)
print(f"Built {len(econstat_snap)} econstat snapshot rows")
print(econstat_snap.groupby("horizon")["is_missing"].mean() * 100)

Built 834 econstat snapshot rows
horizon
14d      0.0
1d       0.0
21d      0.0
3d       0.0
7d       0.0
final    0.0
Name: is_missing, dtype: float64


## Match months between the two series

In [ ]:
MONTHS = {"JAN":1,"FEB":2,"MAR":3,"APR":4,"MAY":5,"JUN":6,"JUL":7,"AUG":8,"SEP":9,"OCT":10,"NOV":11,"DEC":12}

def ticker_to_month_key(ticker):
    m = re.search(r"-(\d{2})([A-Z]{3})$", ticker)
    if not m:
        return None
    yy, mon = m.group(1), m.group(2)
    if mon not in MONTHS:
        return None
    return f"{2000 + int(yy)}-{MONTHS[mon]:02d}"

kxcpiyoy_adj = pd.read_csv(os.path.join(OUTPUT_DIR, "forecast_panel_adjusted.csv"))
kxcpiyoy_events = kxcpiyoy_adj["event_ticker"].unique()
kxcpiyoy_month_map = {ticker_to_month_key(t): t for t in kxcpiyoy_events if ticker_to_month_key(t)}

econstat_events_list = econstat_snap["event_ticker"].unique()
econstat_month_map = {ticker_to_month_key(t): t for t in econstat_events_list if ticker_to_month_key(t)}

overlap_months = sorted(set(kxcpiyoy_month_map) & set(econstat_month_map))
print(f"Overlapping release months: {len(overlap_months)}")
print(overlap_months)

Overlapping release months: 7
['2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06']


## Full comparison across ALL horizons

For each overlapping month AND each horizon: derive KXCPIYOY's differenced bin probability at that horizon, and compare against KXECONSTATCPIYOY's directly-quoted bin probability at the SAME horizon (not just 'final' this time).

In [ ]:
ALL_HORIZONS = ["21d", "14d", "7d", "3d", "1d", "final"]
comparison_rows = []

for month in overlap_months:
    kx_event = kxcpiyoy_month_map[month]
    ec_event = econstat_month_map[month]

    for horizon in ALL_HORIZONS:
        kx_curve = kxcpiyoy_adj[(kxcpiyoy_adj["event_ticker"] == kx_event) &
                                  (kxcpiyoy_adj["horizon"] == horizon)].sort_values("threshold")
        if len(kx_curve) < 2:
            continue
        thresholds = kx_curve["threshold"].values
        s_adj = kx_curve["S_adjusted"].values
        bin_probs = -1 * (s_adj[1:] - s_adj[:-1])

        ec_rows = econstat_snap[(econstat_snap["event_ticker"] == ec_event) &
                                  (econstat_snap["horizon"] == horizon) &
                                  (~econstat_snap["is_missing"])]
        if ec_rows.empty:
            continue

        for i in range(len(thresholds) - 1):
            bin_low, bin_high = thresholds[i], thresholds[i + 1]
            kx_prob = bin_probs[i]
            match = ec_rows[(ec_rows["threshold"] >= bin_low) & (ec_rows["threshold"] < bin_high)]
            if match.empty:
                continue
            ec_prob = match.iloc[0]["mid_quote"]
            comparison_rows.append({
                "month": month, "horizon": horizon, "kx_event": kx_event, "ec_event": ec_event,
                "bin_low": bin_low, "bin_high": bin_high,
                "kxcpiyoy_prob": kx_prob, "kxeconstat_prob": ec_prob,
                "abs_diff": abs(kx_prob - ec_prob),
            })

full_comparison_df = pd.DataFrame(comparison_rows)
full_comparison_df.to_csv(os.path.join(OUTPUT_DIR, "cross_series_comparison_full.csv"), index=False)
print(f"Built {len(full_comparison_df)} bin-level comparisons across "
      f"{full_comparison_df['month'].nunique()} months x {full_comparison_df['horizon'].nunique()} horizons")

Built 624 bin-level comparisons across 7 months x 6 horizons


## Does agreement improve as release approaches? (core question)

In [ ]:
summary = full_comparison_df.groupby("horizon").agg(
    avg_abs_diff=("abs_diff", "mean"),
    n_comparisons=("abs_diff", "size"),
).reindex(ALL_HORIZONS)

corr_by_horizon = full_comparison_df.groupby("horizon").apply(
    lambda g: g["kxcpiyoy_prob"].corr(g["kxeconstat_prob"]) if len(g) > 2 else None
).reindex(ALL_HORIZONS)
summary["correlation"] = corr_by_horizon

print(summary)
print()
print("Overall avg abs diff:", round(full_comparison_df["abs_diff"].mean(), 4))
print("Overall correlation:", round(full_comparison_df["kxcpiyoy_prob"].corr(full_comparison_df["kxeconstat_prob"]), 4))

         avg_abs_diff  n_comparisons  correlation
horizon                                          
21d          0.052328            104     0.680848
14d          0.050955            104     0.680379
7d           0.055516            104     0.686481
3d           0.057404            104     0.667626
1d           0.063576            104     0.625102
final        0.063576            104     0.625102

Overall avg abs diff: 0.0572
Overall correlation: 0.6575


/var/folders/vb/m7hw60gj479gs79ks94n24nr0000gn/T/ipykernel_39841/3597384301.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  corr_by_horizon = full_comparison_df.groupby("horizon").apply(


In [ ]:
# flag any event/horizon combos where the econstat side looks flat/uninformative
# (all bins near 0.5 -> likely wide-spread placeholder quotes, like KXECONSTATCPIYOY-25DEC)
flat_check = full_comparison_df.groupby(["month", "horizon"])["kxeconstat_prob"].std().reset_index()
flat_check.columns = ["month", "horizon", "econstat_prob_std"]
likely_flat = flat_check[flat_check["econstat_prob_std"] < 0.02]
print("Event/horizon combos where econstat probabilities look suspiciously flat (likely thin/placeholder quotes):")
print(likely_flat)

Event/horizon combos where econstat probabilities look suspiciously flat (likely thin/placeholder quotes):
Empty DataFrame
Columns: [month, horizon, econstat_prob_std]
Index: []
